# Build one ore-processing learning example

**Question:** What physical labels and interpolation errors does a small simulator-trained Ridge model produce?

Level: 1. Prerequisites: None; begin here. Reading time: 20 minutes.



## Objectives

The entry lesson constructs physical feature/target provenance and fits Ridge recovery on a 48/16 interpolation split of 64 simulated states. The three case probes establish the interpretation of labels before one model and one validation question are examined.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Import the pinned analysis tools once and resolve inputs relative to the notebook folder.

In [ ]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Data

Owner: OreFlow authors, MIT project licence. This notebook reads `data/derived/contract/operating_contract.json`; `pipeline.cases.catalog` supplies the four authored case definitions. Production `build_design` generates the 64 labels through the circuit engine. It does not read scores from learning.json or load deployed surrogate models. The operating states and ore factors are synthetic model inputs, not fabricated measurements. Source: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

The first regression task uses recovery as its sole fitted target. Four authored cases contribute sixteen engine states each; production features describe ore and operating conditions without an explicit case identifier. Ridge learns from 48 interpolation-training rows after StandardScaler fits those rows alone. The remaining sixteen rows are predicted once and supply RMSE, MAE and R2. Three separate nominal circuit solves establish where labels originate. Case identity can still be encoded indirectly by physical features, so this within-design test concerns interpolation among represented cases rather than a new ore population.

Create a first supervised dataset from sixty-four bounded engine solves. Four authored circuits contribute sixteen states each. The feature builder uses physical ore and operating descriptors and excludes the case identifier. The target builder returns recovery, log-upgrade and total specific energy. This lesson fits recovery only, keeping one understandable prediction task while showing where the other labels originate. The state generator enforces the operating contract, and its fixed seed makes the input design repeatable.

In [ ]:
# Build the learning design from physical circuit outputs.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.learning import build_design, interpolation_split, leave_one_case_out
from pipeline.methods.learning import leave_one_group_out, transfer_groups, FEATURES, TARGETS
import contextlib
import io

contract = json.loads(
    (repo / "data/derived/contract/operating_contract.json").read_text(encoding="utf-8")
)
case_ids = ("copper_porphyry_soft", "copper_porphyry_hard", "copper_oxide", "iron_magnetite_fine")
selected = tuple((CASE_BY_ID[name] for name in case_ids))
with contextlib.redirect_stdout(io.StringIO()):
    design = build_design(contract, cases=selected, per_case=16, seed=20261031)
x, y, case_of = (design["x"], design["y"], design["case"])
print("Design rows:", len(x))
print("Features:", len(FEATURES))
print("Targets:", TARGETS)


Inspect a nominal soft-porphyry label directly. Recovery is a fraction of valuable material recovered, whereas specific energy is normalized by treated mass. A learner sees both only if the analysis explicitly choose them as targets; here it receives recovery. This separate circuit solve demonstrates the label's physical provenance before the matrix is passed into a statistical estimator. It also establishes that there is no measured plant response hidden behind the word training.

In [ ]:
# Inspect the nominal label of copper porphyry soft before pooling its rows.
case = CASE_BY_ID["copper_porphyry_soft"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Reference case:", case.id)
print("Reference recovery (%):", round(reference.metrics["recovery_pct"], 3))
print("Reference energy (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Resolve the oxide-copper case with its own nominal definition. Its mineral properties differ from porphyry, so two nearby operating settings need not imply nearby recovery labels across the cases. Case identity is absent from the feature vector, but physical properties can still encode strong case signatures. This is why a good random-row test cannot automatically be called generalization to an unfamiliar mineral.

In [ ]:
# Inspect the nominal label of copper oxide before pooling its rows.
case = CASE_BY_ID["copper_oxide"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Reference case:", case.id)
print("Reference recovery (%):", round(reference.metrics["recovery_pct"], 3))
print("Reference energy (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Compute the magnetite example using its separation flowsheet rather than forcing it through a flotation-only model. The resulting feature vector includes indicators of the installed process stages. Comparing this label with the copper examples teaches what a common supervised representation can and cannot mean. Shared recovery units permit one regression target, but do not make the underlying mineral and separation mechanisms identical.

In [ ]:
# Inspect the nominal label of iron magnetite fine before pooling its rows.
case = CASE_BY_ID["iron_magnetite_fine"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Reference case:", case.id)
print("Reference recovery (%):", round(reference.metrics["recovery_pct"], 3))
print("Reference energy (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Reserve one quarter of each case's rows with the production interpolation splitter. Training and test both contain all four cases, so this first experiment asks about unseen operating states among represented circuits. The printed case inventory exposes that limitation before any scores appear. Scaling belongs in the training pipeline: using the full design's means would allow test information to influence the fitted representation even without leaking a target value.

In [ ]:
# Construct three validation questions on exactly the same design.
train, test = interpolation_split(case_of, test_fraction=0.25, seed=20261031)
case_fold = next((f for f in leave_one_case_out(case_of) if f[0] == "copper_porphyry_soft"))
group_fold = leave_one_group_out(case_of, transfer_groups(selected))[0]
splits = {
    "Interpolation": (train, test),
    "Case holdout": case_fold[1:],
    "Ore-group holdout": group_fold[1:],
}
for protocol, (training, testing) in splits.items():
    print(protocol, len(training), len(testing))
    print("Training cases:", sorted(set(case_of[training].tolist())))
    print("Test cases:", sorted(set(case_of[testing].tolist())))
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

scores, predictions = ([], {})
splits = {list(splits)[0]: next(iter(splits.values()))}


Fit Ridge after training-only standardization and predict the reserved recovery labels. Its coefficient penalty stabilizes a linear fit when physical features are correlated. Keep alpha fixed rather than searching it against the displayed test scores. Compute RMSE, MAE and R² directly from the predictions. R² can be negative if the model performs worse than a test-mean reference, which is an informative result rather than a reason to hide a fold.

In [ ]:
# Ridge penalizes squared coefficients after scaling.
for protocol, (training, testing) in splits.items():
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    model.fit(x[training], y[training, 0])
    predicted = model.predict(x[testing])
    print("Prediction range (%):", np.round([predicted.min(), predicted.max()], 3))
    print("Reference range (%):", np.round([y[testing, 0].min(), y[testing, 0].max()], 3))
    predictions["Ridge", protocol] = (testing, predicted)
    scores.append(
        {
            "Model": "Ridge",
            "Protocol": protocol,
            "RMSE (pp)": float(np.sqrt(mean_squared_error(y[testing, 0], predicted))),
            "MAE (pp)": float(mean_absolute_error(y[testing, 0], predicted)),
            "R2": float(r2_score(y[testing, 0], predicted)),
        }
    )
display(pd.DataFrame(scores).tail(3).round(3))


Draw the interpolation error for this single estimator. There is one validation protocol and one fitted model in the entry lesson, making the bar a summary of a concrete prediction calculation rather than a literature ranking. The plot uses percentage points because it measures recovery differences, not relative percent change. Read it alongside the displayed prediction and target ranges to judge whether the magnitude is meaningful.

In [ ]:
# Compare recovery RMSE within each validation question.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
score = pd.DataFrame(scores)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
score.pivot(index="Model", columns="Protocol", values="RMSE (pp)").plot.barh(ax=ax)
ax.set(xlabel="Recovery RMSE\n(pp)", ylabel="", title="Reference validation error")
ax.tick_params(axis="x", rotation=0)
handles, legend_names = ax.get_legend_handles_labels()
legend_short = {"Case holdout": "Case", "Ore-group holdout": "Ore group"}
ax.legend(
    handles,
    [legend_short.get(name, name) for name in legend_names],
    fontsize=18,
    ncol=2,
    **legend_style,
)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)
print("Computed RMSE rows:", len(score))


The error bar is descriptive and has no confidence interval because this experiment uses one deterministic split. Its sixty-four labels were generated from four shared case definitions, so they are not independent geological observations. The model has learned an approximation to the process engine. Before adding nonlinear estimators, first confirm the label units, the case representation in both partitions and the fact that the standardizer saw training rows only.

## Results

**Answer:** The common design contains <!-- result:Design rows -->0<!-- /result --> simulated states. Ridge fits <!-- result:Training rows -->0<!-- /result --> training rows and predicts <!-- result:Interpolation test rows -->0<!-- /result --> reserved rows, with RMSE <!-- result:Ridge interpolation RMSE (pp) -->0<!-- /result --> pp. Every case remains represented during training; this result establishes interpolation on engine-generated labels, not prediction for an absent ore or a measured plant.

Collect actual design, training and interpolation-test counts together with the Ridge recovery RMSE.

In [ ]:
# Summarize the computed outcomes used in this lesson's interpretation.
score_by_key = {(row["Model"], row["Protocol"]): row for row in scores}
answer_values = {
    "Design rows": len(x),
    "Ridge interpolation RMSE (pp)": score_by_key["Ridge", "Interpolation"]["RMSE (pp)"],
}
answer_values.update({"Training rows": len(train), "Interpolation test rows": len(test)})
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Summary

The entry fit uses 48 training rows to predict sixteen reserved rows from a 64-state simulator design. Label provenance and units precede any model ranking. No ore case is absent from training in this interpolation question.

## References

- https://doi.org/10.1023/A:1010933404324
- https://doi.org/10.1023/A:1010933404324